# BugBuster — Desktop App Launcher

**Architecture:** Tauri v2 + Leptos (WASM via `wasm32-unknown-unknown`) · Rust workspace · keychain-backed token migration  
**Last refreshed:** 2026-05-04

Builds and launches the BugBuster desktop application.

| Section | Action |
|---|---|
| 1 | Toolchain check |
| 2 | WASM frontend check (`cargo check --target wasm32-unknown-unknown`) |
| 3 | Tauri host check (`cargo check` in src-tauri) |
| 4 | Dev mode launch instructions (run in terminal — blocking) |
| 5 | Connect via Python library once desktop is running |
| 6 | Keychain token inspection |

> `cargo tauri dev` is **blocking** and opens a native window. Run it in a
> separate terminal, not inside a notebook cell.

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO_ROOT = Path(os.getcwd()).parent
APP_DIR   = REPO_ROOT / 'DesktopApp' / 'BugBuster'
TAURI_DIR = APP_DIR / 'src-tauri'

# Locate cargo — may not be on PATH in macOS notebooks
CARGO = 'cargo'
if not subprocess.run(['cargo', '--version'], capture_output=True).returncode == 0:
    candidate = Path.home() / '.cargo' / 'bin' / 'cargo'
    if candidate.exists():
        CARGO = str(candidate)

print(f'Repo root  : {REPO_ROOT}')
print(f'App dir    : {APP_DIR}')
print(f'Tauri dir  : {TAURI_DIR}')
print(f'App exists : {APP_DIR.exists()}')

## 1. Toolchain check

In [ ]:
def _ver(cmd):
    try:
        r = subprocess.run(cmd, capture_output=True, text=True, timeout=15)
        out = (r.stdout + r.stderr).strip()
        return out.splitlines()[0][:70] if out else '?'
    except Exception:
        return None

checks = [
    ('rustc',        [CARGO.replace('cargo', 'rustc'), '--version']),
    ('cargo',        [CARGO, '--version']),
    ('cargo-tauri',  [CARGO, 'tauri', '--version']),
    ('wasm target',  None),  # checked separately
]

all_ok = True
for name, cmd in checks:
    if cmd is None:
        continue
    ver = _ver(cmd)
    status = 'OK' if ver else 'MISSING'
    print(f'  {status}  {name}: {ver or ""}')
    if not ver:
        all_ok = False

# Check wasm32 target
rustup = subprocess.run(['rustup', 'target', 'list', '--installed'],
                        capture_output=True, text=True)
if rustup.returncode == 0 and 'wasm32-unknown-unknown' in rustup.stdout:
    print('  OK  wasm32-unknown-unknown: installed')
else:
    print('  MISSING  wasm32-unknown-unknown: run: rustup target add wasm32-unknown-unknown')
    all_ok = False

print()
print('Toolchain ready.' if all_ok else 'Install missing tools before continuing.')

## 2. WASM frontend check

Checks the Leptos frontend crate compiles to WASM. This is a `cargo check` (fast)
— it does not produce a binary.

In [ ]:
print('Running: cargo check --target wasm32-unknown-unknown')
r = subprocess.run(
    [CARGO, 'check', '--target', 'wasm32-unknown-unknown'],
    cwd=str(APP_DIR),
    capture_output=True,
    text=True,
)
# cargo check sends diagnostics to stderr
output = (r.stdout + r.stderr).strip()
# Show last 40 lines to keep output manageable
lines = output.splitlines()
if len(lines) > 40:
    print(f'  ... ({len(lines) - 40} lines omitted) ...')
    lines = lines[-40:]
print('\n'.join(lines))
print(f'\nExit code: {r.returncode}')

## 3. Tauri host check

Checks the native Tauri host (Rust, native target) compiles cleanly.

In [ ]:
print('Running: cargo check (src-tauri)')
r = subprocess.run(
    [CARGO, 'check'],
    cwd=str(TAURI_DIR),
    capture_output=True,
    text=True,
)
output = (r.stdout + r.stderr).strip()
lines = output.splitlines()
if len(lines) > 40:
    print(f'  ... ({len(lines) - 40} lines omitted) ...')
    lines = lines[-40:]
print('\n'.join(lines))
print(f'\nExit code: {r.returncode}')

## 4. Launch dev mode

`cargo tauri dev` starts both the Leptos WASM frontend (via trunk) and the Tauri
native window. It is **blocking** and must run in a separate terminal.

```bash
# Run in a separate terminal from the repo root:
cd DesktopApp/BugBuster
cargo tauri dev
```

For a release build:

```bash
cd DesktopApp/BugBuster
cargo tauri build
# Output: src-tauri/target/release/bundle/
```

In [ ]:
# Run in shell (see markdown above):
# cargo tauri dev
print('Launch cargo tauri dev in a separate terminal — see markdown cell above.')

## 5. Connect via Python library (desktop running)

Once the desktop app is running, it exposes an HTTP server on localhost.
Connect with `connect_http()` to verify the app is live.

In [ ]:
import sys, os
sys.path.insert(0, str(REPO_ROOT / 'python'))
import bugbuster

# Default port used by the desktop app's embedded HTTP server
HTTP_HOST = os.environ.get('BUGBUSTER_HTTP_HOST', 'localhost')
HTTP_PORT = int(os.environ.get('BUGBUSTER_HTTP_PORT', '8080'))

bb_desktop = None

try:
    bb_desktop = bugbuster.connect_http(HTTP_HOST, port=HTTP_PORT)
    fw = bb_desktop.get_firmware_version()
    mac = bb_desktop.get_mac_address()
    print(f'Connected to desktop app at {HTTP_HOST}:{HTTP_PORT}')
    print(f'Firmware: {fw[0]}.{fw[1]}.{fw[2]}')
    print(f'MAC     : {mac}')
except Exception as e:
    print(f'[Desktop not running or not accessible] {e}')
    print(f'Start the app with: cargo tauri dev')
    print(f'Then re-run this cell.')

## 6. Keychain token inspection

The desktop app uses keychain-backed token storage (item migrated in v1.0.0).
On macOS the admin token is stored in the system Keychain under the service
`BugBuster`. The cell below retrieves it via the `security` CLI for inspection.

In [ ]:
import platform, subprocess

if platform.system() == 'Darwin':
    # macOS keychain lookup (prompts for permission if protected)
    r = subprocess.run(
        ['security', 'find-generic-password', '-s', 'BugBuster', '-w'],
        capture_output=True, text=True,
    )
    if r.returncode == 0:
        token = r.stdout.strip()
        masked = token[:4] + '...' + token[-4:] if len(token) > 8 else '****'
        print(f'Admin token found in Keychain: {masked}')
    else:
        print('Admin token not found in Keychain.')
        print('Pair the device via the desktop app first (Settings -> Pair).')
elif platform.system() == 'Windows':
    print('Windows credential store: use Credential Manager or the Rust `keyring` crate API.')
else:
    print('Linux: secret stored via libsecret / GNOME Keyring.')
    print('Use: secret-tool lookup service BugBuster')

# Also show token via Python library if connected
if bb_desktop is not None:
    try:
        token = bb_desktop.get_admin_token()
        masked = token[:4] + '...' + token[-4:] if len(token) > 8 else '****'
        print(f'Admin token via USB: {masked}')
    except Exception as e:
        print(f'Could not fetch token via library: {e}')

In [ ]:
# Clean up
if bb_desktop is not None:
    try:
        bb_desktop.disconnect()
    except Exception:
        pass
print('Done.')